# 03 — Z_cap Synthesis: R8 + Trio Rollup

Math×code anchor for SaaS-Builder Stage-2 — Phase 3 Task 3.4 of the
notebooks-trio plan. Loads the committed C4 emit (Z_cap_pinned.json)
and verifies:

- **R8**: Z_cap closed form with strictly-positive 95% credible-interval
  lower bound

The final cell aggregates ALL 8 verdicts (R1–R8) into a single
TrioRollup — the canonical end-of-trio summary across the entire
math anchor.

Verify-only: no PyMC re-fit, <10s wall. Tamper anchor surfaced
on every cell via `loader.trio_audit_sha256`.

In [1]:
from simulations.notebooks.saas_builder_stage_2.env import (
    DATA_ROOT, reproducibility_seed,
)
from simulations.saas_builder.verify import (
    CommittedArtifactLoader,
    RTagVerdict,
    TrioRollup,
    verify_r1_sigma_0_anchor,
    verify_r2_kappa_eliminated_in_pi_t,
    verify_r3_perpetual_identity,
    verify_r4_bracket_cardinality,
    verify_r5_marginalization_match,
    verify_r6_softplus_l1_tightness,
    verify_r7_s_t_pin,
    verify_r8_z_cap_closed_form,
)

reproducibility_seed()
loader = CommittedArtifactLoader(DATA_ROOT)
print(f"Trio audit anchor: {loader.trio_audit_sha256}")

Trio audit anchor: 147e01dfcbda6a1b776beed403891ce5a1a5fefdd186c6dcf60e95e4b480e0c2


## §4 — R8: Z_cap pinned (cohort cap closed form)

**Decision-citation block**

- **Reference**: verdict memo §1 (Z_cap pinned at 4,687.94 COP/mo, 95% CI [168.17, 14,606.14]); `notes/STAGE_2_RESULTS.md` §4 line 204
- **Why**: R8 is the headline cap pin — the cohort hedge cap derived from R1 (σ₀) → R2 (π(t)) → C4 emit
- **Relevance**: positive 95% CI lower bound is the credible-interval-level confidence statement; without it the cap is statistically zero
- **Connection**: closes the chain R1 → R2 → R3 → R6 → R7 → R8; gates the Stage-3 path-to-deployment

$$
\boxed{
  \mathbb{E}\!\left[\frac{q_t^{\text{det}}}{(X/Y)_t} + \pi(t)\right] \le Z_{\text{cap}} \;=\; 4{,}687.94 \text{ COP/mo}
}
\tag{R8}
$$

$$
\text{95\% CI}_{Z_{\text{cap}}} \;=\; [168.17,\; 14{,}606.14], \qquad
\text{CI lower bound} \;>\; 0
$$

The cohort cap is **strictly positive at credible-interval-level confidence**
under the prior + model. R8 verifies (a) the point estimate matches the
verdict-memo §1 pin within 1 COP, AND (b) the 95% CI lower bound is > 0.

In [2]:
zcap = loader.load_z_cap_pinned()
v8 = verify_r8_z_cap_closed_form(
    z_cap_pinned=zcap, audit_sha256=loader.trio_audit_sha256,
)
assert v8.passed, v8.message
print(f"{v8.r_tag}: Z_cap = {v8.actual:,.2f} COP/mo  "
      f"(expected ≈ {v8.expected:.2f}; residual {v8.residual:.2e})")
print(f"        95% CI lo = {zcap.ci_95_lo:.2f} > 0  "
      f"→ cohort cap strictly positive at CI-level confidence")

R8: Z_cap = 4,687.94 COP/mo  (expected ≈ 4687.94; residual 2.35e-03)
        95% CI lo = 168.17 > 0  → cohort cap strictly positive at CI-level confidence


The committed `Z_cap_pinned.json` matches the verdict-memo §1 pin within
the 1 COP tolerance (residual under 0.1 COP). The 95% CI lower bound is
strictly positive (168.17 COP/mo), so the cohort cap is statistically
distinguishable from zero under the prior + model. R8 closes the chain
R1 → R2 → R3 → R6 → R7 → R8.

Per spec/memo: this verdict is the entry gate for Stage-3 (real-data
conditioning + on-chain Panoptic deployment) per the
`Stage-3 readiness` section of the verdict memo §9.

## §5 — Cross-trio TrioRollup: R1–R8

Self-contained re-run of all 8 verifiers (R1–R8) in this notebook — does
not depend on `v1`–`v7` from notebooks 01/02. The TrioRollup aggregates the
canonical end-of-trio summary across the entire math anchor.

In [3]:
# Re-run all 8 verifiers for a self-contained cross-trio rollup
sha = loader.trio_audit_sha256

v1 = verify_r1_sigma_0_anchor(
    x_bar=2.0, y_bar=1.0, eps=200.0, expected=20_000.0, tol=1e-6,
    audit_sha256=sha,
)
v2 = verify_r2_kappa_eliminated_in_pi_t(audit_sha256=sha)
v3 = verify_r3_perpetual_identity(tol=1e-8, audit_sha256=sha)
brackets = [
    (t, a, c, k)
    for t in range(3) for a in range(2) for c in range(2) for k in range(2)
]
v4 = verify_r4_bracket_cardinality(brackets=brackets, audit_sha256=sha)
chain = loader.load_posterior_chain()
v5 = verify_r5_marginalization_match(
    posterior_chain=chain, tol=200.0, audit_sha256=sha,
)
v6 = verify_r6_softplus_l1_tightness(
    kappa=1.0, tol_factor=1e-3, audit_sha256=sha,
)
revenue = loader.load_revenue_form_verdict()
v7 = verify_r7_s_t_pin(revenue_form_verdict=revenue, audit_sha256=sha)
# v8 already computed above

verdicts: tuple[RTagVerdict, ...] = (v1, v2, v3, v4, v5, v6, v7, v8)
rollup = TrioRollup(
    verdicts=verdicts,
    all_passed=all(v.passed for v in verdicts),
    audit_sha256=sha,
)

print()
print("=" * 70)
print(f"  STAGE-2 MATH ANCHOR — Cross-trio TrioRollup (R1–R8)")
print(f"  audit_sha256: {rollup.audit_sha256}")
print(f"  all_passed:   {rollup.all_passed}")
print("=" * 70)
for v in rollup.verdicts:
    status = "PASS" if v.passed else "FAIL"
    print(f"  {v.r_tag:3s}  {status:4s}  {v.message[:80]}")
print("=" * 70)
print()
assert rollup.all_passed, "Cross-trio R1–R8 rollup FAILED"


  STAGE-2 MATH ANCHOR — Cross-trio TrioRollup (R1–R8)
  audit_sha256: 147e01dfcbda6a1b776beed403891ce5a1a5fefdd186c6dcf60e95e4b480e0c2
  all_passed:   True
  R1   PASS  σ₀ closed form: expected 20000, got 20000, residual 0.00e+00 (tol 1.00e-06)
  R2   PASS  κ ∉ free_symbols(π(t)): True; free_symbols={omega, X_bar, eps, Y_bar, t, sigma_0
  R3   PASS  perpetual identity residual: 0.00e+00 (tol 1.00e-08)
  R4   PASS  |B|=24; factorization tiers=3 α=2 cache=2 κ=2 (must be 3×2×2×2)
  R5   PASS  posterior max MC SE = 1.42e+02 at N=2184000 (tol 2.00e+02)
  R6   PASS  softplus L¹ deviation = 6.579637e-04; threshold = 1.000000e-03 (0.001·κ at κ=1.0
  R7   PASS  runtime HALT-on-flip safeguard armed: True; spec-level S_t pin (Beta(4.5, 95.5))
  R8   PASS  Z_cap = 4687.94 COP/mo (expected ≈ 4687.94; residual 2.35e-03); 95% CI lo = 168.

